# Variant Effect Prediction with NVIDIA Parabricks and CodonFM

## Check GPU Resources

Run only one GPU course at a time on this workshop VM.

1. **Save and close the previous kernel.** Save the previous course's notebook, then shut down its kernel in **Running Terminals and Kernels** in that course's JupyterLab. Closing a browser tab does not release GPU memory.
2. **Check GPU use.** Open **File → New → Terminal** and run `bootcamp-resources status`.
3. **Prepare this course.** If any BioNeMo models are running, run `bootcamp-resources stop --confirm`.
4. **Confirm readiness.** Run `bootcamp-resources status` again. Continue only when the BioNeMo models are stopped and `gpu_processes` is empty.

For Single-cell / Genomics, keep only one notebook kernel open at a time and use the kernel specified in the course guide.

For the complete instructions, see `00_START_HERE.txt` in the course root directory.



#### Authors: Gary Burnett, Fan Cao 

This lab demonstrates an end-to-end computational biology pipeline for predicting the effects of genetic variants on protein-coding genes. We will leverage NVIDIA's GPU-accelerated tools and state-of-the-art deep learning models to process genomic data efficiently.

### What We'll Do

1. **Variant Calling**: We'll start with raw DNA sequencing data (FASTQ files) from an Illumina sequencer and use NVIDIA Parabricks to rapidly align these reads to the human reference genome and identify genetic variants.

2. **Gene Annotation**: We'll process the GENCODE gene annotation database to extract protein-coding sequences and map our detected variants onto specific genes and transcripts.

3. **Effect Prediction**: Finally, we'll use CodonFM, NVIDIA's foundation model for RNA sequence analysis, to predict how each variant affects the protein-coding potential of genes.

### Why This Matters

Genetic variants can have vastly different effects on human health. Some variants are harmless (synonymous), while others can significantly disrupt protein function. This pipeline combines high-performance computing with machine learning to quickly and accurately assess variant effects—a critical capability for clinical genomics and precision medicine.

### Technology Stack

- **NVIDIA Parabricks**: GPU-accelerated genomics tools for variant calling
- **CodonFM**: A foundation model trained on codon-level sequences for fine-grained variant effect prediction
- **Polars**: Efficient DataFrame processing for large genomic datasets
- **BCFTools & SAMTools**: Standard bioinformatics utilities for working with genomic data formats

Let's get started!

## 1. Variant Calling 

<p><img src="../images/variant_calling.png"></p>

### The dataset

First let's download the dataset. The files are described in the next section. 

In [ ]:
import os
from pathlib import Path

# A writable path on your own GPU VM; override DATA_DIR before launching Jupyter.
os.environ.setdefault("DATA_DIR", str((Path.cwd().parent / "data").resolve()))
print("DATA_DIR:", os.environ["DATA_DIR"])


In [ ]:
%%sh

bash ../scripts/download_vep_safe.sh "$DATA_DIR"

In this first section of the notebook, we start with FASTQ files directly off a sequencer and we want to accurately find where any genetic mutations may exist in this sample. The samples in this notebook are from an Illumina short-read sequencer, meaning that the DNA was cut into fragments of about 100 base pairs (bp) each and then run through the sequencer to determine which bases (A,T,C,G) are at which position. This information is stored in a FASTQ file, a human readable format that contains repeated groups of four lines containing a header, the sequence, a separator, and a quality score for each fragment in the sequence. We can open our FASTQ file below to see for ourselves: 

In [ ]:
%%sh 

zcat $DATA_DIR/sample_1.fq.gz | head 

For more information on the FASTQ format, see this paper from [Nucleic Acids Research](https://academic.oup.com/nar/article/38/6/1767/3112533?login=false) describing the initial implementation and modern updates to the format.

Sequencing DNA as fragments is a compromise, since DNA strands are so long and we do not currently have the technology to sequence the entire strand at once. Therefore, the first step in our analysis is to re-assemble these fragments back into their full sequence. This process is called alignment and requires a reference genome in order to know what order to re-assemble the fragments into. Since the reference is used as a "ground truth" sequence, we must be certain that the sequence is correct. There are several options for reference genomes to use, but a common one is the the Genome Reference Consortium Human Build 38 (GRCh38), which is a highly accurate reference genome developed by an international collaboration of researchers from NCI, Sanger, EBI, and others. 

The reference files exist in `data/ref`. There is a `.fasta` file (similar to our `.fastq` samples above) and a set of index files generated by [`bwa index`](https://bio-bwa.sourceforge.net/bwa.shtml) and [`samtools faidx`](https://www.htslib.org/doc/samtools-faidx.html) which will make searching through the reference much more efficient. These operations can take a long time to run, so they are typically pre-computed and downloaded to save time. 

In [ ]:
%%sh 

ls -lh $DATA_DIR/ref

### Alignment and variant calling methods

There are several algorithms that can perform alignment, but we will stick to the commonly used [BWA aligner](https://github.com/lh3/bwa) based on the [Burrows-Wheeler Transform](https://sandbox.bio/concepts/bwt). For specific implementation details see the [original paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC2705234/). 

Once the sample is aligned, we can use a variant caller to determine which locations in the sample differ from the reference genome. In this lab we use DeepVariant as our variant caller because it is highly accurate. DeepVariant was [originaly developed by Google](https://research.google/blog/deepvariant-highly-accurate-genomes-with-deep-neural-networks/) in 2017 as a method to use deep learning for variant calling. It works by converting the reads into images, and then using a convolutional neural network (CNN) to predict variants. This method won multiple awards for best accuracy during the 2020 Precision FDA Truth challenge. 

### GPU-accelerated implementation

We will use GPU-accelerated versions of BWA and DeepVariant through [NVIDIA Parabricks](https://docs.nvidia.com/clara/parabricks/latest/index.html). Parabricks is a suite of GPU-accelerated tools for genomics analysis. It is designed as a drop-in replacement for many of the CPU versions of tools that are commonly used today. The accuracy is guaranteed to be the same as these tools, but because they run on the GPU they can be orders of magnitude faster. This is especially helpful when you have large amounts of data that need to be processed. 

The Parabricks `deepvariant_germline` pipeline as shown in the diagram below takes in FASTQ files and a reference as input, runs GPU-accelerated BWA and then GPU-accelerated DeepVariant all in one step. This combined command allows for compute to be shared between multiple steps, ensuring faster processing that running the tools as separate commands. 

<p><img src="../images/dv_diagram.webp"></p>

*Source: Parabricks `deepvariant_germline` [documentation](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_deepvariant_germline.html)*

Use the next cell to run this pipeline and generate the variants we will use downstream. 

In [ ]:
# Self-study: this kernel runs on your GPU VM host, where Docker is available.
# Clusters without Docker: ask your agent/operator to adapt this same fixed job
# to the site's GPU allocation and supported container runtime; see README.md.
import os
import subprocess

subprocess.run(["bash", "../scripts/run_parabricks.sh", os.environ["DATA_DIR"]], check=True)


The output of DeepVariant is a Variant Call Format (VCF) file which represents all the variants found in the sample in a tabular format. The column names are standardized as follows: 

| Column Name | Value |
|---|---|
| CHROM | Reference sequence name (chromosome or contig) |
| POS | 1-based leftmost position of the variant on CHROM |
| ID | Variant identifier(s) (e.g., rsID) or `.` if none |
| REF | Reference allele sequence at POS |
| ALT | Comma-separated alternate allele(s) observed at POS |
| QUAL | Phred-scaled quality score for the assertion of the ALT allele(s) |
| FILTER | `PASS` if variant passed filters, or semicolon-separated failing filter names |
| INFO | Semicolon-separated annotations (flags or key=value pairs) with additional site-level metadata |
| FORMAT | Colon-separated keys defining the per-sample fields (e.g., `GT:DP:AD`) |
| sample | Sample-specific values corresponding to FORMAT (e.g., genotype, depth, allele depths)

This file also contains a header with meta data about the samples. For more information on the VCF file format, review the [spec](https://samtools.github.io/hts-specs/VCFv4.2.pdf). 

[bcftools](https://github.com/samtools/bcftools) is a powerful command line utility to view and manipulate vcf files. Use [bcftools view](https://samtools.github.io/bcftools/bcftools.html#view) to inspect the contents of our VCF file. 

In [ ]:
%%sh

# Look at the first few lines of the VCF file (without the header) 
bcftools view --no-header $DATA_DIR/out/deepvariant.vcf.gz | head

### Filter VCF

For our task, the VCF includes a lot of unnecessary information. Let's subset the VCF to only include columns we will need later, and to only include rows that pass all the filter criteria. 

To achieve this we use [bcftools query](https://samtools.github.io/bcftools/bcftools.html#query) with the following arguments: 

| Argument | Value |
|---|---|
| `-H` | Print the header. |
| `-f` | Only include certain columns. |
| `-i` | Only include sites that satisfy this criteria. |

Then we zip the output and store it as a new tab-separated file.  

In [ ]:
%%sh 

bcftools query -H -f '%CHROM\t%POS\t%REF\t%ALT\n' \
  -i 'TYPE="snp" && FILTER="PASS"' \
  "${DATA_DIR}/out/deepvariant.vcf.gz" | 
  bgzip > $DATA_DIR/out/deepvariant.filtered.tsv.gz

This new file should have fewer columns: 

In [ ]:
%%sh 

zcat $DATA_DIR/out/deepvariant.filtered.tsv.gz | head

## 2. Gene Annotation

<p><img src="../images/gene_annotation.png"></p>

The [GENCODE project](https://www.gencodegenes.org/pages/gencode.html) was launched by the National Human Genome Research Institute (NHGRI) and aims to annotate all the functional elements in the human genome. It is widely used for bioinformatics and accepted as a gold standard annotation. The annotations are stored in Gene Transfer Format (GTF) format. Each line in a GTF file has the following 9 fields: 

| Field | Description |
|---|---|
| seqname | the chromosome name |
| source | the program that generated the features |
| feature | feature type (e.g., gene, transcript, exon, CDS, UTR, variation) |
| start | 1-based start position of the feature |
| end | 1-based end position of the feature |
| score | floating point score (use `.` if not applicable) |
| strand | `+` for forward strand, `-` for reverse strand |
| frame | reading frame: 0, 1, or 2 indicating which base is the start of the codon (use `.` if not applicable) |
| attribute | semicolon-separated key=value (or key "value") pairs with additional info (e.g., `gene_id`, `transcript_id`, `gene_name`) 

In this notebook, we use the GENCODE Human Release 47 for GRCh38 basic annotation set. Read more about this file on the [GENCODE website](https://www.gencodegenes.org/human/release_47.html). 

In [ ]:
%%sh 

GENCODE_FILE_URL="https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_47/gencode.v47.basic.annotation.gtf.gz"

# Download the gtf file 
wget -nc --progress=bar:force 2>&1 -P $DATA_DIR/ref ${GENCODE_FILE_URL}

Inspect the contents of the GTF file. 

In [ ]:
%%sh 

# Inspect the GTF file 
zcat $DATA_DIR/ref/gencode.v47.basic.annotation.gtf.gz | head 

In `scripts.py` there is a script called `process_gtf_file` which filters for protein-coding genes only and outputs a tab-separated `.tsv` file. The important parts of this file for our purposes include the gene names with their locations in the reference genome. 

In [ ]:
import os
import sys

sys.path.append(os.path.abspath('..'))
from scripts.gtf_processing import process_gtf_file 

REFERENCE_DIR = os.path.join(os.environ['DATA_DIR'], "ref") 
gtf_file = f"{REFERENCE_DIR}/gencode.v47.basic.annotation.gtf.gz"
out_file = f"{REFERENCE_DIR}/gencode.v47.basic.annotation.processed.tsv"

_ = process_gtf_file(gtf_file, out_file)

Open the processed `GTF` file 

In [ ]:
%%sh 

head $DATA_DIR/ref/gencode.v47.basic.annotation.processed.tsv

### Extract coding sequence (CDS)

`extract_cds` is a function that uses the CDS start and stop positions from the GTF file and maps them into the reference file to extract the full CDS sequence. It will also gather information about start and stop codons. 

In [ ]:
from scripts.extract_cds import extract_cds

reference_genome = f"{REFERENCE_DIR}/Homo_sapiens_assembly38.fasta"
annotation_file = f"{REFERENCE_DIR}/gencode.v47.basic.annotation.processed.tsv"
filtered_transcripts_file = f"{REFERENCE_DIR}/gencode.v47.basic.annotation.processed.filtered.tsv"

extract_cds(reference_genome, annotation_file, filtered_transcripts_file)

Open the filtered `.tsv` file and notice the additional columns for information about the CDS sequence and codons. 

In [ ]:
%%sh 

head $DATA_DIR/ref/gencode.v47.basic.annotation.processed.filtered.tsv

### Put it all together 

Lastly, use Python to load this data into a [Polars dataframe](https://docs.pola.rs/py-polars/html/reference/dataframe/index.html). Polars is like Pandas except much more memory efficient for large datasets due to using the Apache Arrow memory format as opposed to Numpy arrays. For more information on the differences, check out the [Polars documentation](https://docs.pola.rs/user-guide/migration/pandas/). 

In [ ]:
import polars as pl 

We have one table that shows the sequence for protein coding region and its index in the genome. 

In [ ]:
gencode_v47_file = os.path.join(REFERENCE_DIR, "gencode.v47.basic.annotation.processed.filtered.tsv")
genes = pl.read_csv(gencode_v47_file, separator="\t")
genes = genes.with_row_index('id')
genes = genes.with_columns(pl.col("id").cast(pl.Int64))
genes.head()

And we have another table with all the variants found in the samples and their index in the genome. 

In [ ]:
variants = pl.read_csv(os.path.join(os.environ['DATA_DIR'], "out/deepvariant.filtered.tsv.gz"), separator='\t', has_header=True)

# Rename columns to simplify (Ex. #[1]CHROM > chrom)
variants.columns = ['chrom', 'pos', 'ref', 'alt'] 

# Remove any positions with multiple alternate alleles 
variants = variants.filter(pl.col('alt').str.contains(r'^[ATCG]$')) 

variants.head()

We can join these tables to find which transcripts have variants and where in the CDS sequence they are. This join requires helper functions. First we use `map_variants_to_genes_by_exons_efficient` to create a map of each transcript and which variants it contains. 

In [ ]:
from scripts.extract_cds import map_variants_to_genes_by_exons_efficient 

# Calculate which genes have variants 
gene_variant_mapping = map_variants_to_genes_by_exons_efficient(genes, variants, variant_columns=['pos', 'ref', 'alt']) 

In [ ]:
gene_variant_mapping

Then we use `convert_gene_variant_mapping_to_df` to put map into a table for easier post-processing. 

In [ ]:
from scripts.extract_cds import convert_gene_variant_mapping_to_df

# Convert map to data frame 
gene_variants = convert_gene_variant_mapping_to_df(gene_variant_mapping, genes, ['pos', 'ref', 'alt'])

# Save the output to a CSV 
gene_variants.write_csv(os.path.join(os.environ['DATA_DIR'], "out/gene_variants.csv"))

gene_variants.head()

We will use this an input to codonFM. 

## 3. Variant Effect Prediction

<p><img src="../images/variant_effect_prediction.png"></p>

CodonFM is an open-source foundation model developed by NVIDIA for RNA sequence analysis and design. It is part of the Clara open model family, which provides tools and models for biology, chemistry, and human health applications. CodonFM is trained on the codons, sets of three nucleotides, which is more fine grained that most amino-acid level protein models. This means it can differentiate between synonymous codons, leading to more accurate results. Check out the CodonFM [blog](https://developer.nvidia.com/blog/introducing-the-codonfm-open-model-for-rna-design-and-analysis), the NVIDIA Digital Biology Examples repo on [GitHub](https://github.com/NVIDIA-Digital-Bio/CodonFM/tree/main), and the codonFM [preprint](https://research.nvidia.com/labs/dbr/assets/data/manuscripts/nv-codonfm-preprint.pdf) for more information. 

### Import libraries

In [ ]:
from scripts.codonfm_helpers import load_dataset
from scripts.codonfm_helpers import run_mutation_predictions
from scripts.codonfm_helpers import load_encodon_inference_model

import os
import torch
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name()}")

### Download the model 

CodonFM comes in three sizes: 80M, 600M, 1B (using random masking), and Cdwt-1B (1B model with Codon-frequency Weighted Masking Strategy). 

In [ ]:
# Set the path to the data directory 
os.environ['MODEL_SIZE'] = "Cdwt-1B"
print("✅ MODEL_SIZE exported to shell")

In [ ]:
%%sh 

cd $DATA_DIR/models && \
    wget -nc --progress=bar:force 2>&1 https://huggingface.co/nvidia/NV-CodonFM-Encodon-${MODEL_SIZE}-v1/resolve/main/NV-CodonFM-Encodon-${MODEL_SIZE}-v1.safetensors && \
    wget -nc --progress=bar:force 2>&1 https://huggingface.co/nvidia/NV-CodonFM-Encodon-${MODEL_SIZE}-v1/resolve/main/config.json

### Load the model onto the GPU

In [ ]:
ckpt_path = os.path.join(os.environ['DATA_DIR'], "models")
checkpoint_paths = {
    "80M": f"{ckpt_path}/NV-CodonFM-Encodon-80M-v1.safetensors",
    "600M": f"{ckpt_path}/NV-CodonFM-Encodon-600M-v1.safetensors",
    "1B": f"{ckpt_path}/NV-CodonFM-Encodon-1B-v1.safetensors",
    "Cdwt-1B": f"{ckpt_path}/NV-CodonFM-Encodon-Cdwt-1B-v1.safetensors"
}

model_loaded = False
encodon_models = {}

for size, checkpoint_path in checkpoint_paths.items():
    if os.path.exists(checkpoint_path):
        try:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            model = load_encodon_inference_model(checkpoint_path, device=device)
            
            # Extract model name from path
            model_name = os.path.basename(os.path.dirname(os.path.dirname(checkpoint_path)))
            display_name = f"EnCodon ({size})"
            
            encodon_models[display_name] = {
                'model': model,
                'path': checkpoint_path,
                'device': device
            }
            print(f"✅ Successfully loaded {display_name} from: {checkpoint_path}")
            model_loaded = True
        except Exception as e:
            print(f"Failed to load from {checkpoint_path}: {e}")
            continue

if not model_loaded:
    print("❌ Could not load any Encodon model from the specified paths.")
    print("Please ensure a checkpoint exists or update the checkpoint_paths list.")
else:
    print(f"\n✅ Loaded {len(encodon_models)} models: {list(encodon_models.keys())}")

### Load the dataset

This is the `results.csv` output from the previous section. 

In [ ]:
# Load ClinVar Alphamissense dataset
gene_variants_path = os.path.join(os.environ['DATA_DIR'], "out/gene_variants.csv")
DATASET_CONFIG = {
    'name': 'FASTQ Dataset',
    'data_path': gene_variants_path,
}

# Load the dataset
dataset = load_dataset(DATASET_CONFIG)
print(f"\n📊 Dataset loaded: {dataset is not None}")

### Run mutation prediction 

With the model and the dataset loaded, we are now able to run the codonFM model with our data. 

In [ ]:
# Run predictions if models and dataset are available
if 'encodon_models' in locals() and 'dataset' in locals() and dataset is not None:
    predictions = run_mutation_predictions(encodon_models, dataset)
    print(f"\n✅ Predictions completed for {len(predictions)} models")
else:
    print("❌ Cannot run predictions - missing models or dataset")
    predictions = {}

This will generate a dictionary of log likelihood ratios (LLR) between the reference codon and the alt codon for each mutation. A positive LLR means the mutation is potentiall disruptive, and a negative LLR suggets it is tolerated.  

In [ ]:
predictions

Finally we can join the predictions with our `gene_variants` table and sort the log likelihood ratio to see which variants have the largest effect on their respective genes. 

In [ ]:
pred_df = pl.DataFrame(predictions[f"EnCodon ({os.environ['MODEL_SIZE']})"], schema=['id', 'likelihood_ratio'])
pred_df = pred_df.with_columns(pl.col("id").cast(gene_variants.schema["id"]))
pred_df = pred_df.join(gene_variants, on="id")
pred_df = pred_df.sort("likelihood_ratio", descending=True)
pred_df.head()

Now we have a `likelihood_ratio` column with the rest of the data in the `gene_variants` table. 

## Conclusion

In this lab we performed an end-to-end variant effect prediction pipeline. Starting from FASTQ samples we ran Parabricks BWA and DeepVariant to align to the reference and detect variants. We then used Python to manipulate a GTF file to show us the protein-coding sequence for each gene. Lastly we fed this information into CodonFM to predict the effect of each variant on the annotated genes. The dataset we used today is small but the tools are capable of handling much larger datasets thanks to GPU acceleration. 

Try running this notebook on your own FASTQ files, with the larger Encodon models, or visit the [NVIDIA BioNeMo Framework recipes](https://github.com/NVIDIA/bionemo-framework/tree/main/bionemo-recipes/recipes/codonfm_ptl_te) to retrain the CodonFM model for even more personalized results.


## Save and Release GPU Resources

1. **Save and shut down.** Save this notebook, then shut down its kernel in **Running Terminals and Kernels**.
2. **Release resources.** In **File → New → Terminal**, run `bootcamp-resources status` to check that this course has released the GPU.
3. **Check before switching.** Run `bootcamp-resources status` and confirm that `gpu_processes` is empty before opening the next course's kernel.

If the notebook's cleanup cell restarts the kernel, shut down that restarted kernel in the Running panel before opening another notebook.

When returning to BioNeMo, follow **Check GPU Resources** at the beginning of its lesson and wait for all three models to become **healthy**.
